# 8 · Widgets

A widget is the control a person edits an input with. Every input declares its own; there is no default per type.

In [ ]:
import json
from typing import Annotated

from conductor import CompiledGraph, Graph, GraphNode, NodeRegistry, Param, Result, Series, Static, widgets, node
from conductor.widgets import AnyWidget, Choice, OperatorChoice
from conductor_nodes.types import Json
from pydantic import TypeAdapter

## Widgets with constraints

The widget goes on the `Param`, beside the `title`, `description` and `show_handle` that belong to the field.

In [ ]:
from conductor import node
PRIORITIES = (
    Choice(id="low", title="Low"),
    Choice(id="normal", title="Normal"),
    Choice(id="high", title="High"),
)


@node("send-email", title="Send Email", description="Send an email", category="mail")
def send_email(
    to: Annotated[str, Param(title="Recipient", widget=widgets.TextWidget(pattern=r"^[^@]+@[^@]+$"))],
    subject: Annotated[str, Param(title="Subject", widget=widgets.TextWidget(max_length=200))],
    body: Annotated[str, Param(title="Body", widget=widgets.Textarea(rows=8, min_length=10))],
    priority: Annotated[str, Param(title="Priority", widget=widgets.Dropdown(choices=PRIORITIES))] = "normal",
    retries: Annotated[float, Param(title="Retry attempts", widget=widgets.Range(min_val=0, max_val=5, step=1))] = 1.0,
) -> Annotated[str, Result(title="Status")]:
    return f"sent: {to=}, {subject=}, {priority=}, {retries=}"


for inp in send_email.versions[1].interface.inputs:
    print(f"  {inp.name:<10} kind={inp.widget.kind:<10} title={inp.title!r}  required={inp.required}")

## Every widget

A handle needs a `DType`; with `show_handle=False` any pydantic type will do. A `Series[X]` input takes `ListWidget`, `Tags`, or no widget when only an edge fills it.

In [ ]:
from conductor import node
OPERATORS = (
    OperatorChoice(id="is-empty", title="is empty", category="text", arity=1),
    OperatorChoice(id="contains", title="contains", category="text", arity=2),
)


@node("widget-showcase", title="Showcase", description="Every widget", category="demo")
def showcase(
    # str & code
    heading: Annotated[str, Param(title="Title", widget=widgets.TextWidget(max_length=100))],
    notes: Annotated[str, Param(title="Notes", widget=widgets.Textarea(rows=6))],
    template: Annotated[str, Param(title="Template", widget=widgets.TemplateTextarea(rows=3))],
    code_snippet: Annotated[str, Param(title="Code", widget=widgets.CodeEditor(language="python"))],
    # Choice
    style: Annotated[str, Param(title="Style", widget=widgets.Dropdown(choices=(Choice(id="a", title="A"), Choice(id="b", title="B"))))],
    user: Annotated[str, Param(title="User", widget=widgets.EntityDropdown(entity_kind="user"))],
    # Numeric
    count: Annotated[float, Param(title="Count", widget=widgets.NumberWidget(integer_only=True, min_val=0))],
    volume: Annotated[float, Param(title="Volume", widget=widgets.Range(min_val=0, max_val=1, step=0.01))],
    # Boolean
    enabled: Annotated[bool, Param(title="Enabled", widget=widgets.Switch())],
    # Date & file
    starts: Annotated[str, Param(title="Starts", widget=widgets.DatePicker(min_date="2024-01-01", seed="today"))],
    upload: Annotated[str, Param(title="Upload", widget=widgets.FileUpload(accept=(".pdf", ".csv"), max_size_mb=10))],
    # Collections
    items: Annotated[Series[str], Param(title="Items", widget=widgets.ListWidget(min_items=1, max_items=10))],
    labels: Annotated[Series[str], Param(title="Labels", widget=widgets.Tags())],
    sources: Annotated[Series[str], Param(title="Sources")],
    # Structured
    table: Annotated[Json, Param(title="Table", widget=widgets.TableInput(min_rows=1, min_columns=2))],
    config: Annotated[dict, Param(title="Config", connectable=False, widget=widgets.SchemaBuilder(schema={"type": "object"}))],
    routing: Annotated[dict, Param(title="Routing", connectable=False, widget=widgets.IfElseBuilder(operators=OPERATORS))],
) -> Annotated[str, Result(title="Summary")]:
    return "all good"


inputs = showcase.versions[1].interface.inputs
print(f"{len(inputs)} inputs, {len({inp.widget.kind for inp in inputs if inp.widget})} distinct widget kinds:\n")
for inp in inputs:
    dtype = getattr(inp.dtype, "id", None) or getattr(inp.dtype, "__name__", inp.dtype)
    handle = "" if inp.connectable else "  (no handle)"
    kind = inp.widget.kind if inp.widget else "— (edges only)"
    print(f"  {inp.name:<13} {dtype:<8} kind={kind}{handle}")

## The dump

`AnyWidget` is the union of every widget, discriminated by `kind`.

In [ ]:
w = widgets.Range(min_val=0, max_val=100, step=5)
print(json.dumps(TypeAdapter(AnyWidget).dump_python(w, mode="json"), indent=2))

In [ ]:
# The same widget as its Input carries it — title and description sit beside it.
from conductor import Input, node

volume = next(inp for inp in inputs if inp.name == "volume")
print(json.dumps(TypeAdapter(Input).dump_python(volume, mode="json"), indent=2))

## Running a graph

The engine validates typed-in values through the declared types.

In [ ]:
from conductor import node
@node("fmt", title="Format", description="Glue", category="demo")
def format(
    n: Annotated[float, Param(title="N", widget=widgets.NumberWidget(integer_only=True))],
    label: Annotated[str, Param(title="Label", widget=widgets.TextWidget())],
) -> Annotated[str, Result(title="Out")]:
    return f"{label}={int(n)}"


reg_run = NodeRegistry([format])

compiled = CompiledGraph.from_graph(
    Graph(nodes=[GraphNode(id="n", type="fmt", version=1, bindings={"n": Static(42), "label": Static("answer")})]),
    reg_run,
)
results = compiled.results(await compiled.run())
print(results["n"]["result"])